In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# ============================================================
# Single-cell: HEO + Rice (Cammeo & Osmancik) + LR tuning
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.exceptions import ConvergenceWarning

# ------------------------------------------------------------
# 1. Benchmark functions (for quick Sphere test)
# ------------------------------------------------------------

def sphere(x):
    return np.sum(x**2)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def schwefel_221(x):
    return np.max(np.abs(x))

def schwefel_222(x):
    absx = np.abs(x)
    return np.sum(absx) + np.prod(absx)

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def bent_cigar(x):
    return x[0]**2 + 1e6*np.sum(x[1:]**2)

def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)

def alpine(x):
    return np.sum(np.abs(x * np.sin(x) + 0.1 * x))

def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0

def rastrigin(x):
    return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))

def ackley(x):
    d = x.size
    a = 20.0
    b = 0.2
    c = 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    term1 = -a * np.exp(-b*np.sqrt(s1/d))
    term2 = -np.exp(s2/d)
    return term1 + term2 + a + np.e

def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r

def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi = x[i]
        xj = x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

# ------------------------------------------------------------
# 2. HEO-style optimizer
# ------------------------------------------------------------

class HEO:
    """
    Halfway-Escape-Optimization-style optimizer (approximation).
    Minimizes f(x) over x in [lower, upper]^dim.
    """
    def __init__(
        self,
        func,
        dim,
        lower=-100.0,
        upper=100.0,
        n_particles=100,
        max_iters=1000,
        seed=None,
        escape_init=0.0,
        escape_increment=0.1,
        escape_max=2.0,
        stagnation_patience_global=30,
        stagnation_patience_vibration=10,
        skip_patience=80,
        w_global=0.5,
        sigma_radius=0.1,
        center_clip_factor=1.0,
        skip_scale=0.5,
    ):
        self.func = func
        self.dim = dim
        if np.isscalar(lower):
            self.lower = np.full(dim, float(lower))
        else:
            self.lower = np.array(lower, dtype=float)
        if np.isscalar(upper):
            self.upper = np.full(dim, float(upper))
        else:
            self.upper = np.array(upper, dtype=float)

        self.n_particles = n_particles
        self.max_iters = max_iters

        self.escape_factor = escape_init
        self.escape_increment = escape_increment
        self.escape_max = escape_max
        self.stagnation_patience_global = stagnation_patience_global
        self.stagnation_patience_vibration = stagnation_patience_vibration
        self.skip_patience = skip_patience
        self.w_global = w_global
        self.sigma_radius = sigma_radius
        self.center_clip_factor = center_clip_factor
        self.skip_scale = skip_scale

        self.rng = np.random.default_rng(seed)

        self.positions = None
        self.pbest = None
        self.pbest_f = None
        self.gbest = None
        self.gbest_f = None
        self.no_improve_local = None
        self.no_improve_global = 0
        self.energy = None
        self.it = 0

    def _init_swarm(self):
        self.positions = self.rng.uniform(
            self.lower, self.upper, size=(self.n_particles, self.dim)
        )
        self.pbest = self.positions.copy()
        self.pbest_f = np.apply_along_axis(self.func, 1, self.pbest)
        best_idx = np.argmin(self.pbest_f)
        self.gbest = self.pbest[best_idx].copy()
        self.gbest_f = self.pbest_f[best_idx]

        self.no_improve_local = np.zeros(self.n_particles, dtype=int)
        self.no_improve_global = 0
        self.energy = np.ones(self.n_particles, dtype=float)
        self.escape_factor = float(self.escape_factor)
        self.it = 0

    def _center_clipping(self, X):
        diffs = np.abs(X - self.gbest)
        spread = np.max(diffs, axis=0)
        r_soft = self.rng.random(self.dim)
        radius = self.center_clip_factor * (spread + 1e-12) * (0.5 + r_soft)
        domain_half = 0.5 * (self.upper - self.lower)
        radius = np.minimum(radius, domain_half)
        group_lower = np.maximum(self.gbest - radius, self.lower)
        group_upper = np.minimum(self.gbest + radius, self.upper)
        return np.clip(X, group_lower, group_upper)

    def _vibration(self, X):
        stagnant_mask = self.no_improve_local >= self.stagnation_patience_vibration
        if not np.any(stagnant_mask):
            return X

        std_vec = np.std(X, axis=0)
        std_vec = np.where(std_vec == 0, 1e-12, std_vec)

        for i in np.where(stagnant_mask)[0]:
            scale = self.sigma_radius * std_vec / (1.0 + self.energy[i])
            noise = self.rng.normal(loc=0.0, scale=scale, size=self.dim)
            X[i] = X[i] + noise
            self.energy[i] *= 1.02

        return X

    def _position_update(self, X):
        new_X = np.empty_like(X)
        for i in range(self.n_particles):
            x = X[i]
            p = self.pbest[i]
            g = self.gbest
            halfway = 0.5 * (p + g)
            dist_p = p - x
            dist_g = g - x
            r_escape = self.rng.exponential(1.0, size=self.dim)
            r_rand = self.rng.exponential(1.0, size=self.dim)

            if self.escape_factor <= 1e-12:
                step = self.w_global * (self.rng.random(self.dim) - 0.5) * (
                    np.abs(dist_p) + np.abs(dist_g)
                )
                x_new = halfway + step
            else:
                dir_vec = x - halfway
                if np.allclose(dir_vec, 0.0):
                    dir_vec = self.rng.normal(size=self.dim)
                step_escape = self.escape_factor * r_escape * dir_vec
                step_noise = self.w_global * r_rand * (dist_p + dist_g)
                x_new = x + step_escape + step_noise

            new_X[i] = x_new
        return new_X

    def _random_skip(self, X):
        if self.no_improve_global < self.skip_patience:
            return X

        domain_range = (self.upper - self.lower)
        radius = self.skip_scale * domain_range
        shift = self.rng.uniform(-radius, radius)
        center = np.clip(self.gbest + shift, self.lower, self.upper)
        low = np.maximum(center - radius, self.lower)
        high = np.minimum(center + radius, self.upper)
        X = self.rng.uniform(low, high, size=(self.n_particles, self.dim))

        self.no_improve_global = 0
        self.no_improve_local[:] = 0
        self.energy[:] = 1.0
        return X

    def run(self, verbose=False):
        self._init_swarm()
        for it in range(self.max_iters):
            self.it = it
            X = self._position_update(self.positions)
            X = self._vibration(X)
            X = self._center_clipping(X)
            X = self._random_skip(X)

            f_vals = np.apply_along_axis(self.func, 1, X)

            improved_local = f_vals < self.pbest_f
            self.pbest[improved_local] = X[improved_local]
            self.pbest_f[improved_local] = f_vals[improved_local]
            self.no_improve_local[improved_local] = 0
            self.no_improve_local[~improved_local] += 1

            best_idx = np.argmin(f_vals)
            best_f = f_vals[best_idx]
            if best_f + 1e-12 < self.gbest_f:
                self.gbest_f = best_f
                self.gbest = X[best_idx].copy()
                self.no_improve_global = 0
            else:
                self.no_improve_global += 1

            if self.no_improve_global >= self.stagnation_patience_global:
                self.escape_factor = min(
                    self.escape_factor + self.escape_increment, self.escape_max
                )
            else:
                self.escape_factor *= 0.99

            self.positions = X

            if verbose and (it % max(1, self.max_iters // 10) == 0):
                print(
                    f"iter={it:4d}, gbest_f={self.gbest_f:.4e}, escape={self.escape_factor:.3f}"
                )

        return self.gbest, self.gbest_f

# ------------------------------------------------------------
# 3. Load Rice dataset (your file)
# ------------------------------------------------------------

csv_path = "/kaggle/input/rice-cammeo-and-osmancik/Rice_data_type.csv"
df_rice = pd.read_csv(csv_path)
print("Using dataset:", csv_path)
print("Dataset shape:", df_rice.shape)
print(df_rice.head())

# last column is class label
target_col = df_rice.columns[-1]

# drop index-like columns explicitly
cols_to_drop = ["Unnamed: 0", "id", "ID"]
feature_cols = [c for c in df_rice.columns if c not in cols_to_drop + [target_col]]

X = df_rice[feature_cols].values.astype(float)
y_raw = df_rice[target_col].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------------------------------------------
# 4. LR eval + HEO optimization
# ------------------------------------------------------------

def lr_eval_from_vector(vec, X_tr, y_tr, X_val, y_val):
    """
    vec[0]: log10(C) in [-3, 3]
    vec[1]: scaled max_iter in [0, 1] -> [100, 2000]
    Returns loss = 1 - F1 (to minimize).
    """
    log10_C = vec[0]
    max_iter = int(100 + vec[1] * (2000 - 100))
    max_iter = max(100, min(max_iter, 2000))
    C = 10 ** log10_C

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        model = LogisticRegression(
            C=C,
            max_iter=max_iter,
            penalty="l2",
            solver="lbfgs",
        )
        model.fit(X_tr, y_tr)

    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    return 1.0 - f1

def heo_optimize_lr(
    X_train_scaled,
    y_train,
    n_particles=50,
    max_iters=50,
    seed=123,
):
    # inner validation split
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train
    )

    lower = np.array([-3.0, 0.0])  # log10(C), scaled_iter
    upper = np.array([3.0, 1.0])

    def objective(v):
        return lr_eval_from_vector(v, X_tr, y_tr, X_val, y_val)

    heo = HEO(
        func=objective,
        dim=2,
        lower=lower,
        upper=upper,
        n_particles=n_particles,
        max_iters=max_iters,
        seed=seed,
        stagnation_patience_global=10,
        stagnation_patience_vibration=5,
        skip_patience=25,
        w_global=0.5,
        sigma_radius=0.1,
        center_clip_factor=1.0,
        skip_scale=0.5,
    )

    best_vec, best_loss = heo.run(verbose=True)

    # decode best hyperparameters consistently with lr_eval_from_vector
    log10_C = best_vec[0]
    max_iter = int(100 + best_vec[1] * (2000 - 100))
    max_iter = max(100, min(max_iter, 2000))
    C = 10 ** log10_C

    best_model = LogisticRegression(
        C=C,
        max_iter=max_iter,
        penalty="l2",
        solver="lbfgs",
    )
    best_model.fit(X_train_scaled, y_train)

    return best_model, best_vec, best_loss

# ------------------------------------------------------------
# 5. Run HEO on Rice dataset
# ------------------------------------------------------------

heo_model, best_vec, best_loss = heo_optimize_lr(
    X_train_scaled, y_train,
    n_particles=50,
    max_iters=50,
    seed=123,
)

print("\n=== HEO-tuned Logistic Regression ===")
print("Best vector (log10(C), scaled_iter):", best_vec)
print("Best validation loss (1 - F1):", best_loss)

y_pred_test = heo_model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)

print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1-score: {f1:.4f}")

# ------------------------------------------------------------
# 6. Quick Sphere test
# ------------------------------------------------------------

def quick_sphere_test():
    heo = HEO(
        func=sphere,
        dim=30,
        lower=-100.0,
        upper=100.0,
        n_particles=30,
        max_iters=200,
        seed=1,
    )
    _, best_f = heo.run(verbose=False)
    print("\nQuick Sphere test (30D) best_f:", best_f)

quick_sphere_test()


Using dataset: /kaggle/input/rice-cammeo-and-osmancik/Rice_data_type.csv
Dataset shape: (3810, 9)
   Unnamed: 0     Area   Perimeter  Major_Axis_Length  Minor_Axis_Length  \
0           0  15231.0  525.578979         229.749878          85.093788   
1           1  14656.0  494.311005         206.020065          91.730972   
2           2  14634.0  501.122009         214.106781          87.768288   
3           3  13176.0  458.342987         193.337387          87.448395   
4           4  14688.0  507.166992         211.743378          89.312454   

   Eccentricity  Convex_Area    Extent      Class  
0      0.928882      15617.0  0.572896  b'Cammeo'  
1      0.895405      15072.0  0.615436  b'Cammeo'  
2      0.912118      14954.0  0.693259  b'Cammeo'  
3      0.891861      13368.0  0.640669  b'Cammeo'  
4      0.906691      15262.0  0.646024  b'Cammeo'  
iter=   0, gbest_f=4.5317e-02, escape=0.000
iter=   5, gbest_f=4.5317e-02, escape=0.000
iter=  10, gbest_f=4.5317e-02, escape=0.200
i

In [1]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1. Benchmark functions (matching Tables 1 & 2, 30D, [-100,100])
# ============================================================

def sphere(x):
    return np.sum(x**2)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def schwefel_221(x):
    return np.max(np.abs(x))

def schwefel_222(x):
    absx = np.abs(x)
    return np.sum(absx) + np.prod(absx)

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def bent_cigar(x):
    return x[0]**2 + 1e6*np.sum(x[1:]**2)

def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)

def alpine(x):
    return np.sum(np.abs(x * np.sin(x) + 0.1 * x))

def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2)/4000.0 - np.prod(np.cos(x/np.sqrt(i))) + 1.0

def rastrigin(x):
    return 10.0*x.size + np.sum(x**2 - 10.0*np.cos(2*np.pi*x))

def ackley(x):
    d = x.size
    a = 20.0
    b = 0.2
    c = 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    term1 = -a * np.exp(-b*np.sqrt(s1/d))
    term2 = -np.exp(s2/d)
    return term1 + term2 + a + np.e

def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1]-1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r

def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi = x[i]
        xj = x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2. Paper-style HEO implementation (Algorithm 1 + eqs)
# ============================================================

class HEO:
    """
    Halfway Escape Optimization (HEO) — implementation close to Algorithm 1
    and equations (1–7), (13–22) in the paper.

    - Position update: eqs (1–3) with r1,r2,r3
    - r1 ~ U(1-R, 1+R), r2 ~ U(0.5,1.5), r3 ~ U(0,1)
    - Global counter c (escape factor) used in position update
    - Per-quantum energy a_i for Vibration (eqs (13–16))
    - Center clipping (eqs (17–19)) using L_inf norm
    - Random Skip (eqs (20–22)) to explore new region
    """

    def __init__(
        self,
        func,
        dim,
        lower=-100.0,
        upper=100.0,
        swarm_size=100,
        max_iters=1000,
        R=1.0,          # controls r1 range: [1-R, 1+R]
        a_max=10.0,     # upper scale for energy a_i
        c_max=30,       # when c > c_max -> random skip
        seed=None,
    ):
        self.func = func
        self.dim = dim
        self.swarm_size = swarm_size
        self.max_iters = max_iters
        self.R = R
        self.a_max = a_max
        self.c_max = c_max

        if np.isscalar(lower):
            self.lower = np.full(dim, float(lower))
        else:
            self.lower = np.array(lower, dtype=float)

        if np.isscalar(upper):
            self.upper = np.full(dim, float(upper))
        else:
            self.upper = np.array(upper, dtype=float)

        self.rng = np.random.default_rng(seed)

        # internal state
        self.positions = None
        self.local_best_pos = None
        self.local_best_cost = None
        self.global_best_pos = None
        self.global_best_cost = None

        self.energy = None  # a_i
        self.c_global = 0   # c_i

    def _init_swarm(self):
        self.positions = self.rng.uniform(
            self.lower, self.upper, size=(self.swarm_size, self.dim)
        )
        # initial personal bests
        self.local_best_pos = self.positions.copy()
        self.local_best_cost = np.apply_along_axis(self.func, 1, self.local_best_pos)

        # initial global best
        idx = np.argmin(self.local_best_cost)
        self.global_best_pos = self.local_best_pos[idx].copy()
        self.global_best_cost = self.local_best_cost[idx]

        # energy levels for vibration
        self.energy = np.zeros(self.swarm_size, dtype=float)

        # global escape counter
        self.c_global = 0

    def _position_update(self, X):
        """
        Position update (eqs (1–3)).
        x_{i+1} = x_i + v_g + v_l
        v_g = (x_g - x_i (c+1) * r1) * r2 * r3
        v_l = (x_l - x_i (c+1) * r1) * r2 * (1 - r3)
        """
        k, d = X.shape

        r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=(k, d))
        r2 = self.rng.uniform(0.5, 1.5, size=(k, d))
        r3 = self.rng.uniform(0.0, 1.0, size=(k, d))

        c_factor = (self.c_global + 1.0)

        xg = self.global_best_pos
        xl = self.local_best_pos

        # broadcasting
        term = X * (c_factor * r1)
        v_g = (xg - term) * r2 * r3
        v_l = (xl - term) * r2 * (1.0 - r3)

        return X + v_g + v_l

    def _vibration(self, X, vib_mask):
        """
        Vibration (eq (13)):
            x_{i+1} = x_i + n / (1 + e^{a_i})
        where n ~ N(0, σ_x), σ_x approximated as std of the swarm.
        """
        if not np.any(vib_mask):
            return X

        # standard deviation across swarm for each dimension
        std_vec = np.std(X, axis=0)
        std_vec = np.where(std_vec == 0.0, 1e-12, std_vec)

        idxs = np.where(vib_mask)[0]
        for idx in idxs:
            # eq16: n ~ N(0, σ_x_i)
            n = self.rng.normal(loc=0.0, scale=std_vec, size=self.dim)
            scale = 1.0 / (1.0 + np.exp(self.energy[idx]))  # 1/(1+e^{a_i})
            X[idx] = X[idx] + scale * n

        return X

    def _center_clipping(self, X):
        """
        Center clipping (eqs (17–19)):
            S_result = S_bound ∩ S_group
            b_y = ||x_i - x_g|| * r5,   r5 ~ U(0,2)
        We use L_inf norm to define b_y as a scalar radius, then clip to a box
        around x_g with half-side b_y.
        """
        k, d = X.shape
        X_clipped = np.empty_like(X)
        for i in range(k):
            xi = X[i]
            # distance from global best
            diff = xi - self.global_best_pos
            # L_inf norm
            dist = np.max(np.abs(diff))
            r5 = self.rng.uniform(0.0, 2.0)
            b_y = dist * r5

            # build group box
            lower_group = self.global_best_pos - b_y
            upper_group = self.global_best_pos + b_y

            # intersect with domain
            lower_res = np.maximum(self.lower, lower_group)
            upper_res = np.minimum(self.upper, upper_group)

            # clip
            X_clipped[i] = np.clip(xi, lower_res, upper_res)

        return X_clipped

    def _random_skip(self, X):
        """
        Random Skip (eqs (20–22)):
            x_i = (x_i + r) / 2
        where r ~ U(lower, upper) component-wise.
        """
        k, d = X.shape
        rand_points = self.rng.uniform(self.lower, self.upper, size=(k, d))
        X_new = 0.5 * (X + rand_points)
        # also clip to bounds for safety
        return np.clip(X_new, self.lower, self.upper)

    def run(self, verbose=False):
        self._init_swarm()

        for it in range(self.max_iters):
            # 1) Position update
            X = self._position_update(self.positions)

            # 2) Evaluate fitness once after position update
            f_vals = np.apply_along_axis(self.func, 1, X)

            # 3) Check global + local improvements
            # Global improvements
            improved_global = f_vals < self.global_best_cost
            if np.any(improved_global):
                # best of the improved
                idx_candidates = np.where(improved_global)[0]
                idx_best = idx_candidates[np.argmin(f_vals[improved_global])]
                self.global_best_cost = float(f_vals[idx_best])
                self.global_best_pos = X[idx_best].copy()
                # halve global escape counter
                self.c_global = int(self.c_global / 2)

            # Local improvements (but not global)
            improved_local = (f_vals < self.local_best_cost) & (~improved_global)
            self.local_best_pos[improved_local] = X[improved_local]
            self.local_best_cost[improved_local] = f_vals[improved_local]
            # lower energy for improved locals (similar to eq (14) second case)
            self.energy[improved_local] = np.floor(self.energy[improved_local] / 2.0)

            # 4) Vibration for non-improved ones
            vib_mask = ~(improved_global | improved_local)
            X = self._vibration(X, vib_mask)

            # 5) Center clipping
            X = self._center_clipping(X)

            # 6) Update energy a_i (eqs (14–15))
            r4 = self.rng.uniform(0.0, 1.0, size=self.swarm_size)
            # increment energy when a_i * r4 < a_max
            inc_mask = (self.energy * r4) < self.a_max
            self.energy[inc_mask] += 1.0

            # 7) Random skip if global counter too large
            if self.c_global > self.c_max:
                X = self._random_skip(X)
                self.c_global = 0  # reset

            # commit positions
            self.positions = X

            # increment global counter
            self.c_global += 1

            if verbose and (it % max(1, self.max_iters // 10) == 0):
                print(f"iter={it:4d}, gbest={self.global_best_cost:.4e}, c={self.c_global}")

        return self.global_best_pos, self.global_best_cost


# ============================================================
# 3. Benchmark driver matching the paper's protocol
# ============================================================

DIM = 30
LOWER = -100.0
UPPER = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30  # paper uses 30 validations

results = []

for fname, f in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO on {fname} ...")

    for run in range(RUNS):
        heo = HEO(
            func=f,
            dim=DIM,
            lower=LOWER,
            upper=UPPER,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            R=1.0,
            a_max=10.0,
            c_max=30,
            seed=run + 1234,  # reproducible runs
        )
        t0 = time.time()
        _, best_f = heo.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals)
    times = np.array(times)

    mean_cost = float(np.mean(best_vals))
    std_cost = float(np.std(best_vals))
    mean_time_1000 = float(np.mean(times))  # seconds per 1000 iters (MAX_ITERS=1000)

    results.append({
        "Function": fname,
        "HEO_mean_cost": mean_cost,
        "HEO_std_cost": std_cost,
        "HEO_time_s_per_1000iters": mean_time_1000,
    })

df_results = pd.DataFrame(results)
display(df_results)


Running HEO on Sphere ...
Running HEO on Step ...
Running HEO on Schwefel 2.21 ...
Running HEO on Schwefel 2.22 ...
Running HEO on Rosenbrock ...
Running HEO on BentCigar ...
Running HEO on Sumsquares2 ...
Running HEO on Alpine ...
Running HEO on Griewank ...
Running HEO on Rastrigin ...
Running HEO on Ackley ...
Running HEO on Levy ...
Running HEO on Salomon ...
Running HEO on Schaffer ...


,Function,HEO_mean_cost,HEO_std_cost,HEO_time_s_per_1000iters
0,Sphere,3.078962e+04,8.800830e+03,4.668788
1,Step,2.995523e+04,7.386342e+03,4.940816
2,Schwefel 2.21,8.128079e+01,5.899624e+00,4.681440
3,Schwefel 2.22,8.858322e+34,4.568265e+35,5.006269
4,Rosenbrock,1.289643e+10,5.856310e+09,5.487784
5,BentCigar,3.023145e+10,8.660940e+09,4.833563
6,Sumsquares2,4.095840e+05,8.453628e+04,4.991137
7,Alpine,4.098465e+02,6.321266e+01,5.115439
8,Griewank,8.697403e+00,2.200207e+00,5.579737
9,Rastrigin,3.147400e+04,8.693386e+03,5.145421


In [1]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1. Benchmark functions (14, 30D, [-100,100])
# ============================================================

def sphere(x):
    return np.sum(x**2)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def schwefel_221(x):
    return np.max(np.abs(x))

def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def bent_cigar(x):
    return x[0]**2 + 1e6 * np.sum(x[1:]**2)

def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)

def alpine(x):
    return np.sum(np.abs(x * np.sin(x) + 0.1 * x))

def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2) / 4000.0 - np.prod(np.cos(x / np.sqrt(i))) + 1.0

def rastrigin(x):
    return 10.0 * x.size + np.sum(x**2 - 10.0 * np.cos(2*np.pi*x))

def ackley(x):
    d = x.size
    a = 20.0
    b = 0.2
    c = 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    term1 = -a * np.exp(-b * np.sqrt(s1/d))
    term2 = -np.exp(s2/d)
    return term1 + term2 + a + np.e

def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1] - 1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r

def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi = x[i]
        xj = x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2. HEO-inspired optimizer (stable, HEO-style behaviour)
# ============================================================

class HEOInspired:
    """
    HEO-inspired optimizer:
      - Halfway move between personal and global best
      - PSO-like attraction to pbest and gbest
      - Vibration for stagnant particles
      - Center clipping around global best
      - Random skip when global stagnation is large
    Continuous minimization on [lower, upper]^dim.
    """
    def __init__(
        self,
        func,
        dim,
        lower=-100.0,
        upper=100.0,
        swarm_size=100,
        max_iters=1000,
        seed=None,
        # exploration/exploitation
        phi_p_max=2.5,
        phi_p_min=0.5,
        phi_g_max=2.5,
        phi_g_min=0.5,
        phi_h_max=1.5,
        phi_h_min=0.3,
        # stagnation control
        stagnation_patience_local=20,
        stagnation_patience_global=40,
        skip_patience=80,
        # vibration / clipping
        vibration_scale=0.2,
        clip_factor=2.0,
        skip_fraction=0.3,
    ):
        self.func = func
        self.dim = dim
        self.swarm_size = swarm_size
        self.max_iters = max_iters
        self.vibration_scale = vibration_scale
        self.clip_factor = clip_factor
        self.skip_fraction = skip_fraction

        if np.isscalar(lower):
            self.lower = np.full(dim, float(lower))
        else:
            self.lower = np.array(lower, dtype=float)

        if np.isscalar(upper):
            self.upper = np.full(dim, float(upper))
        else:
            self.upper = np.array(upper, dtype=float)

        self.rng = np.random.default_rng(seed)

        # coefficients schedule
        self.phi_p_max = phi_p_max
        self.phi_p_min = phi_p_min
        self.phi_g_max = phi_g_max
        self.phi_g_min = phi_g_min
        self.phi_h_max = phi_h_max
        self.phi_h_min = phi_h_min

        self.stagnation_patience_local = stagnation_patience_local
        self.stagnation_patience_global = stagnation_patience_global
        self.skip_patience = skip_patience

        # internal state
        self.X = None
        self.pbest = None
        self.pbest_f = None
        self.gbest = None
        self.gbest_f = None
        self.no_improve_local = None
        self.no_improve_global = 0

    def _init_swarm(self):
        self.X = self.rng.uniform(
            self.lower, self.upper, size=(self.swarm_size, self.dim)
        )
        self.pbest = self.X.copy()
        self.pbest_f = np.apply_along_axis(self.func, 1, self.pbest)
        idx = np.argmin(self.pbest_f)
        self.gbest = self.pbest[idx].copy()
        self.gbest_f = float(self.pbest_f[idx])
        self.no_improve_local = np.zeros(self.swarm_size, dtype=int)
        self.no_improve_global = 0

    def _coeff_schedule(self, t):
        """Linear decay of phi_p, phi_g, phi_h over iterations."""
        T = max(1, self.max_iters - 1)
        alpha = t / T
        phi_p = self.phi_p_max - alpha * (self.phi_p_max - self.phi_p_min)
        phi_g = self.phi_g_max - alpha * (self.phi_g_max - self.phi_g_min)
        phi_h = self.phi_h_max - alpha * (self.phi_h_max - self.phi_h_min)
        return phi_p, phi_g, phi_h

    def _position_update(self, t):
        """PSO-like + halfway move; stable step sizes."""
        phi_p, phi_g, phi_h = self._coeff_schedule(t)
        r_p = self.rng.random((self.swarm_size, self.dim))
        r_g = self.rng.random((self.swarm_size, self.dim))
        r_h = self.rng.random((self.swarm_size, self.dim))

        # broadcast global best
        g_mat = np.broadcast_to(self.gbest, (self.swarm_size, self.dim))
        p_mat = self.pbest

        halfway = 0.5 * (p_mat + g_mat)

        # main update
        step_p = phi_p * r_p * (p_mat - self.X)
        step_g = phi_g * r_g * (g_mat - self.X)
        step_h = phi_h * r_h * (halfway - self.X)

        X_new = self.X + step_p + step_g + step_h

        # clip to global bounds
        X_new = np.clip(X_new, self.lower, self.upper)
        return X_new

    def _vibration(self, X):
        """Shake particles that haven't improved locally for a while."""
        stagnant = self.no_improve_local >= self.stagnation_patience_local
        if not np.any(stagnant):
            return X

        # global std to scale noise
        std_vec = np.std(X, axis=0)
        std_vec = np.where(std_vec == 0.0, 1e-12, std_vec)

        idxs = np.where(stagnant)[0]
        for i in idxs:
            noise = self.rng.normal(
                loc=0.0,
                scale=self.vibration_scale * std_vec,
                size=self.dim,
            )
            X[i] = X[i] + noise

        X = np.clip(X, self.lower, self.upper)
        return X

    def _center_clipping(self, X):
        """Restrict particles to a box around global best."""
        # approximate spread
        diffs = np.abs(X - self.gbest)
        spread = np.max(diffs, axis=0)  # per-dimension max
        # radius proportional to spread but not smaller than a bit of domain
        domain_span = self.upper - self.lower
        min_radius = 0.05 * domain_span
        radius = np.maximum(self.clip_factor * spread, min_radius)

        box_lower = np.maximum(self.gbest - radius, self.lower)
        box_upper = np.minimum(self.gbest + radius, self.upper)

        # clip each particle to this "group" box intersected with global bounds
        return np.clip(X, box_lower, box_upper)

    def _random_skip(self, X):
        """Randomly re-init a fraction of worst particles when stuck."""
        if self.no_improve_global < self.skip_patience:
            return X

        # number of particles to re-init
        k = int(self.skip_fraction * self.swarm_size)
        if k <= 0:
            return X

        # indices of worst k particles
        worst_idx = np.argsort(self.pbest_f)[-k:]
        # random around global best within domain
        span = (self.upper - self.lower)
        rand_center = self.gbest
        low = np.maximum(rand_center - span * 0.5, self.lower)
        high = np.minimum(rand_center + span * 0.5, self.upper)
        X[worst_idx] = self.rng.uniform(low, high, size=(k, self.dim))

        # reset their local bests
        self.pbest[worst_idx] = X[worst_idx]
        self.pbest_f[worst_idx] = np.apply_along_axis(self.func, 1, self.pbest[worst_idx])
        self.no_improve_local[worst_idx] = 0

        # reset global stagnation counter
        self.no_improve_global = 0

        return X

    def run(self, verbose=False):
        self._init_swarm()
        for t in range(self.max_iters):
            # 1) halfway/PSO-like update
            X_new = self._position_update(t)

            # 2) vibration for stagnant locals
            X_new = self._vibration(X_new)

            # 3) center clipping around global best
            X_new = self._center_clipping(X_new)

            # 4) evaluate
            f_vals = np.apply_along_axis(self.func, 1, X_new)

            # 5) update local bests
            improved_local = f_vals < self.pbest_f
            self.pbest[improved_local] = X_new[improved_local]
            self.pbest_f[improved_local] = f_vals[improved_local]
            self.no_improve_local[improved_local] = 0
            self.no_improve_local[~improved_local] += 1

            # 6) update global best
            idx = np.argmin(f_vals)
            best_f = f_vals[idx]
            if best_f + 1e-12 < self.gbest_f:
                self.gbest_f = float(best_f)
                self.gbest = X_new[idx].copy()
                self.no_improve_global = 0
            else:
                self.no_improve_global += 1

            # 7) random skip if heavily stuck
            X_new = self._random_skip(X_new)

            self.X = X_new

            if verbose and (t % max(1, self.max_iters // 10) == 0):
                print(f"iter={t:4d}, gbest={self.gbest_f:.4e}")

        return self.gbest, self.gbest_f

# ============================================================
# 3. Benchmark protocol (same structure as paper)
# ============================================================

DIM = 30
LOWER = -100.0
UPPER = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30   # you can set to 10 if Kaggle is too slow

results = []

for fname, f in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO-inspired on {fname} ...")
    for run in range(RUNS):
        heo = HEOInspired(
            func=f,
            dim=DIM,
            lower=LOWER,
            upper=UPPER,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            seed=run + 1234,
        )
        t0 = time.time()
        _, best_f = heo.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals)
    times = np.array(times)

    results.append({
        "Function": fname,
        "HEOInspired_mean_cost": float(best_vals.mean()),
        "HEOInspired_std_cost": float(best_vals.std()),
        "HEOInspired_time_s_per_1000iters": float(times.mean()),
    })

df_results = pd.DataFrame(results)
display(df_results)


Running HEO-inspired on Sphere ...
Running HEO-inspired on Step ...
Running HEO-inspired on Schwefel 2.21 ...
Running HEO-inspired on Schwefel 2.22 ...
Running HEO-inspired on Rosenbrock ...
Running HEO-inspired on BentCigar ...
Running HEO-inspired on Sumsquares2 ...
Running HEO-inspired on Alpine ...
Running HEO-inspired on Griewank ...
Running HEO-inspired on Rastrigin ...
Running HEO-inspired on Ackley ...
Running HEO-inspired on Levy ...
Running HEO-inspired on Salomon ...
Running HEO-inspired on Schaffer ...


,Function,HEOInspired_mean_cost,HEOInspired_std_cost,HEOInspired_time_s_per_1000iters
0,Sphere,6.682997e+02,2.495033e+03,1.287405
1,Step,3.466667e+00,2.883671e+00,1.808704
2,Schwefel 2.21,2.800706e+01,5.829600e+00,1.320671
3,Schwefel 2.22,1.271386e+03,1.569108e+02,1.898523
4,Rosenbrock,1.251010e+05,2.833743e+05,1.820165
5,BentCigar,4.411672e+06,8.650264e+06,1.294167
6,Sumsquares2,1.656136e+04,2.218781e+04,1.463091
7,Alpine,1.246877e+02,6.813117e+01,1.904755
8,Griewank,1.784408e-01,3.864301e-01,1.911683
9,Rastrigin,2.649439e+02,9.649102e+01,1.685737


In [2]:
import numpy as np
import pandas as pd
import time

# ============================================================
# 1. Benchmark functions (unimodal + multimodal)
# ============================================================

def sphere(x):
    return np.sum(x**2)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def schwefel_221(x):
    return np.max(np.abs(x))

def schwefel_222(x):
    a = np.abs(x)
    return np.sum(a) + np.prod(a)

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def bent_cigar(x):
    return x[0]**2 + 1e6 * np.sum(x[1:]**2)

def sumsquares2(x):
    i = np.arange(1, x.size+1)
    return np.sum(i * x**2)

def alpine(x):
    return np.sum(np.abs(x * np.sin(x) + 0.1 * x))

def griewank(x):
    i = np.arange(1, x.size+1)
    return np.sum(x**2) / 4000.0 - np.prod(np.cos(x / np.sqrt(i))) + 1.0

def rastrigin(x):
    return 10.0 * x.size + np.sum(x**2 - 10.0 * np.cos(2*np.pi*x))

def ackley(x):
    d = x.size
    a = 20.0
    b = 0.2
    c = 2*np.pi
    s1 = np.sum(x**2)
    s2 = np.sum(np.cos(c*x))
    term1 = -a * np.exp(-b * np.sqrt(s1/d))
    term2 = -np.exp(s2/d)
    return term1 + term2 + a + np.e

def levy(x):
    w = 1 + (x - 1)/4
    term1 = np.sin(np.pi*w[0])**2
    term3 = (w[-1] - 1)**2 * (1 + np.sin(2*np.pi*w[-1])**2)
    term2 = np.sum((w[:-1] - 1)**2 * (1 + 10*np.sin(np.pi*w[:-1] + 1)**2))
    return term1 + term2 + term3

def salomon(x):
    r = np.sqrt(np.sum(x**2))
    return 1 - np.cos(2*np.pi*r) + 0.1*r

def schaffer(x):
    total = 0.0
    for i in range(x.size - 1):
        xi = x[i]
        xj = x[i+1]
        num = np.sin(np.sqrt(xi**2 + xj**2))**2 - 0.5
        den = (1 + 0.001*(xi**2 + xj**2))**2
        total += 0.5 + num/den
    return total

benchmark_funcs = [
    ("Sphere", sphere),
    ("Step", step),
    ("Schwefel 2.21", schwefel_221),
    ("Schwefel 2.22", schwefel_222),
    ("Rosenbrock", rosenbrock),
    ("BentCigar", bent_cigar),
    ("Sumsquares2", sumsquares2),
    ("Alpine", alpine),
    ("Griewank", griewank),
    ("Rastrigin", rastrigin),
    ("Ackley", ackley),
    ("Levy", levy),
    ("Salomon", salomon),
    ("Schaffer", schaffer),
]

# ============================================================
# 2. HEO implementation (Algorithm 1 + eqs (1)-(7),(13)-(22))
# ============================================================

class HEO:
    """
    Halfway Escape Optimization (HEO) as literally as possible from:

      Algorithm 1 (p.7)
      Position update eqs (1)-(3), random r1,r2,r3 eqs (4)-(6), escape c_i eq (7)
      Vibration eqs (13)-(16)
      Center clipping eqs (17)-(19)
      Random skip eqs (20)-(22)

    Assumptions where the paper is ambiguous are kept minimal and commented.
    """

    def __init__(
        self,
        func,
        dim,
        bound=100.0,
        swarm_size=100,
        max_iters=1000,
        a_max=10,
        c_max=30,
        R=1.0,
        seed=None,
    ):
        self.func = func
        self.dim = dim
        self.bound = float(bound)
        self.swarm_size = swarm_size
        self.max_iters = max_iters
        self.a_max = float(a_max)
        self.c_max = int(c_max)
        self.R = float(R)

        self.lower = -self.bound * np.ones(dim)
        self.upper = self.bound * np.ones(dim)

        self.rng = np.random.default_rng(seed)

        # Swarm state
        self.X = None               # positions
        self.local_best = None      # personal best positions
        self.local_best_f = None    # personal best fitness
        self.a = None               # energy levels a_i per quantum
        self.c = 0                  # escape counter c_i (global)
        self.global_best = None
        self.global_best_f = None

    def _init_swarm(self):
        # Positions uniform in [-bound, bound]^dim
        self.X = self.rng.uniform(self.lower, self.upper,
                                  size=(self.swarm_size, self.dim))
        # Initial personal bests
        self.local_best = self.X.copy()
        self.local_best_f = np.apply_along_axis(self.func, 1, self.local_best)

        # Global best
        idx = np.argmin(self.local_best_f)
        self.global_best = self.local_best[idx].copy()
        self.global_best_f = float(self.local_best_f[idx])

        # Energy levels start at 0
        self.a = np.zeros(self.swarm_size, dtype=float)

        # Escape counter c_i = 0
        self.c = 0

    def _position_update_single(self, x, x_l):
        """
        Position update for one quantum, using eqs. (1)-(3):

          x_{i+1} = x_i + v_g + v_l
          v_g = (x_g - x_i (c_i + 1) * r1) * r2 * r3
          v_l = (x_l - x_i (c_i + 1) * r1) * r2 * (1 - r3)

        We treat r1,r2,r3 as vectors in R^dim, drawn per quantum per iteration.
        """
        x_g = self.global_best

        r1 = self.rng.uniform(1.0 - self.R, 1.0 + self.R, size=self.dim)  # eq (4)
        r2 = self.rng.uniform(0.5, 1.5, size=self.dim)                    # eq (5)
        r3 = self.rng.uniform(0.0, 1.0, size=self.dim)                    # eq (6)

        # term x_i (c_i+1) * r1
        term = x * ((self.c + 1.0) * r1)

        v_g = (x_g - term) * r2 * r3
        v_l = (x_l - term) * r2 * (1.0 - r3)

        x_new = x + v_g + v_l
        # Always enforce search bound S_bound
        x_new = np.clip(x_new, self.lower, self.upper)
        return x_new

    def _vibration(self, x_current, sigma_vec, a_i):
        """
        Vibration step for one quantum (eq. (13)-(16)):

          x_{i+1} = x_i + n / (1 + e^{a_i}), where n ~ N(0, σ_x)
        """
        # n ~ N(0, σ_x) component-wise
        n = self.rng.normal(loc=0.0, scale=sigma_vec, size=self.dim)
        scale = 1.0 / (1.0 + np.exp(a_i))
        x_new = x_current + scale * n
        x_new = np.clip(x_new, self.lower, self.upper)
        return x_new

    def _center_clipping(self, x_current):
        """
        Center clipping (eq. (17)-(19)):

          S_result = S_bound ∩ S_group
          b_y = ||x_i - x_g||_2 * r5,   r5 ~ U(0,2)
          S_group: hypercube centered at x_g with half-side-length b_y
        """
        diff = x_current - self.global_best
        dist = np.linalg.norm(diff, ord=2)
        r5 = self.rng.uniform(0.0, 2.0)  # eq. (19)
        b_y = dist * r5                  # eq. (18)

        lower_group = self.global_best - b_y
        upper_group = self.global_best + b_y

        lower_res = np.maximum(self.lower, lower_group)
        upper_res = np.minimum(self.upper, upper_group)

        x_clipped = np.clip(x_current, lower_res, upper_res)
        return x_clipped

    def _random_skip_all(self):
        """
        Random skip (eq. (20)-(22)), applied to ALL quantums when c_i > c_max:

          x_i = (x_i + r_vec) / 2
          r_j ~ U(0, b),  b = bound
        """
        b = self.bound
        rand_vecs = self.rng.uniform(0.0, b, size=(self.swarm_size, self.dim))
        self.X = 0.5 * (self.X + rand_vecs)
        self.X = np.clip(self.X, self.lower, self.upper)
        # Typically leave personal/global bests; paper says "group random skip"
        # Algorithm 1 resets c_i to 0
        self.c = 0

    def run(self, verbose=False):
        self._init_swarm()

        for it in range(self.max_iters):
            # Compute standard deviation of positions for vibration step
            sigma_vec = np.std(self.X, axis=0)
            sigma_vec = np.where(sigma_vec == 0.0, 1e-12, sigma_vec)

            # Work with a copy for consistent "x_i" in this iteration
            X_old = self.X.copy()

            for j in range(self.swarm_size):
                x_old = X_old[j]
                x_l   = self.local_best[j]
                a_i   = self.a[j]

                # --- Position update (eqs (1)-(3)) ---
                x_pu = self._position_update_single(x_old, x_l)
                f_q  = self.func(x_pu)

                # --- Global best update (Algorithm 1) ---
                if f_q < self.global_best_f:
                    self.global_best_f = float(f_q)
                    self.global_best   = x_pu.copy()
                    self.X[j]          = x_pu
                    self.local_best[j] = x_pu
                    self.local_best_f[j] = f_q
                    # c_i <- int(c_i / 2)
                    self.c = int(self.c / 2)

                # --- Local best update (Algorithm 1) ---
                elif f_q < self.local_best_f[j]:
                    self.local_best[j] = x_pu
                    self.local_best_f[j] = f_q
                    self.X[j] = x_pu
                    # a_i <- int(a_i / 2)
                    self.a[j] = float(int(a_i / 2))

                else:
                    # --- Vibration (eq. (13)-(16)) ---
                    x_vib = self._vibration(x_pu, sigma_vec, a_i)
                    self.X[j] = x_vib

                # --- Center clipping (eq. (17)-(19)) ---
                self.X[j] = self._center_clipping(self.X[j])

                # --- Energy level increment (eq. (14)-(15)) ---
                r4 = self.rng.uniform(0.0, 1.0)  # eq. (15)
                # Only increment if a_i * r4 < a_max
                if self.a[j] * r4 < self.a_max:
                    self.a[j] += 1.0

            # --- Random skip if c_i > c_max (Algorithm 1 + eq. (20)-(22)) ---
            if self.c > self.c_max:
                self._random_skip_all()

            # global escape counter increments each iteration (Algorithm 1, c_i++)
            self.c += 1

            if verbose and (it % max(1, self.max_iters // 10) == 0):
                print(f"iter={it:4d}, gbest={self.global_best_f:.4e}, c={self.c}")

        return self.global_best, self.global_best_f

# ============================================================
# 3. Benchmark: 14 functions, dim=30, bound=100, swarm=100, iters=1000
# ============================================================

DIM = 30
BOUND = 100.0
SWARM_SIZE = 100
MAX_ITERS = 1000
RUNS = 30   # can reduce to 10 if this is too slow

results = []

for fname, f in benchmark_funcs:
    best_vals = []
    times = []
    print(f"Running HEO (paper-style) on {fname} ...")
    for run in range(RUNS):
        heo = HEO(
            func=f,
            dim=DIM,
            bound=BOUND,
            swarm_size=SWARM_SIZE,
            max_iters=MAX_ITERS,
            a_max=10,
            c_max=30,
            R=1.0,
            seed=run + 1234,
        )
        t0 = time.time()
        _, best_f = heo.run(verbose=False)
        t1 = time.time()

        best_vals.append(best_f)
        times.append(t1 - t0)

    best_vals = np.array(best_vals)
    times = np.array(times)

    results.append({
        "Function": fname,
        "HEO_mean_cost": float(best_vals.mean()),
        "HEO_std_cost": float(best_vals.std()),
        "HEO_time_s_per_1000iters": float(times.mean()),
    })

df_results = pd.DataFrame(results)
display(df_results)


Running HEO (paper-style) on Sphere ...
Running HEO (paper-style) on Step ...
Running HEO (paper-style) on Schwefel 2.21 ...
Running HEO (paper-style) on Schwefel 2.22 ...
Running HEO (paper-style) on Rosenbrock ...
Running HEO (paper-style) on BentCigar ...
Running HEO (paper-style) on Sumsquares2 ...
Running HEO (paper-style) on Alpine ...
Running HEO (paper-style) on Griewank ...
Running HEO (paper-style) on Rastrigin ...
Running HEO (paper-style) on Ackley ...
Running HEO (paper-style) on Levy ...
Running HEO (paper-style) on Salomon ...
Running HEO (paper-style) on Schaffer ...


,Function,HEO_mean_cost,HEO_std_cost,HEO_time_s_per_1000iters
0,Sphere,1.436590e+04,8.987893e+03,5.530274
1,Step,1.653943e+04,8.883850e+03,5.772451
2,Schwefel 2.21,7.516948e+01,7.690821e+00,5.725624
3,Schwefel 2.22,3.364400e+20,1.807850e+21,5.860593
4,Rosenbrock,7.278684e+09,4.453192e+09,6.366749
5,BentCigar,1.503379e+10,7.935924e+09,5.785605
6,Sumsquares2,2.466113e+05,1.117137e+05,6.003420
7,Alpine,2.962410e+02,9.734256e+01,5.877689
8,Griewank,4.591475e+00,2.246972e+00,6.547911
9,Rastrigin,1.553795e+04,8.954084e+03,6.046337
